# Export Face Clustering Data for ML Training

This notebook exports face clustering results to CSV files for training a logistic regression merge classifier.

**Outputs**:
- `faces.csv` - one row per face with metadata
- `clusters.csv` - one row per cluster with statistics
- `candidate_pairs.csv` - cluster pair features for ML training

**Pipeline**: Load embeddings → Quality gating → kNN graph → Connected components → Exemplar selection → Compute stats → Generate pairs → Export CSVs

## Setup & Imports

In [ ]:
import sys
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import logging
from PIL import Image
import json
from datetime import datetime

# Auto-reload modules for development
%load_ext autoreload
%autoreload 2

# Add project root to path
sys.path.insert(0, str(Path.cwd().parent))

# Import face_cluster library
from face_cluster import (
    PipelineConfig,
    QualityGater,
    KNNGraphBuilder,
    ConnectedComponentsClusterer,
    D10ExemplarSelector,
    FaceRecord,
)
from face_cluster.viz import (
    plot_distance_matrix,
    print_cluster_summary,
    show_face_grid,
)

# Setup logging
logging.basicConfig(level=logging.INFO, format='%(levelname)s: %(message)s')
logger = logging.getLogger(__name__)

print("Imports complete")

## Configuration

**IMPORTANT**: Edit these parameters for your dataset!

In [ ]:
# ========== DATA SOURCE CONFIGURATION ==========

# Input: embeddings from benchmark results
EMBEDDINGS_FILE = Path("../results/face_clustering_benchmark/embeddings_2026-02-16_00-44-28.npy")

# Output directory
OUTPUT_DIR = Path("../results/face_clustering_training/dataset1")

# Enable pose estimation (requires sixdrepnet)
ESTIMATE_POSE = False  # Set to True for pose filtering

# ========== CLUSTERING PARAMETERS ==========

# Core clustering
K = 5                        # Number of nearest neighbors
DISTANCE_THRESHOLD = 0.35    # Max cosine distance for edges
MIN_CLUSTER_SIZE = 2         # Min cluster size

# Quality gating
BLUR_MIN = 50.0              # Min blur score
YAW_MAX = 30.0               # Max yaw angle (degrees)
PITCH_MAX = 25.0             # Max pitch angle (degrees)
ROLL_MAX = 25.0              # Max roll angle (degrees)

# Exemplar selection
D10_K = 3                    # K for d10 computation
EXEMPLAR_THRESHOLD = 0.35    # Max d10 to be exemplar

# Candidate pair filtering
CANDIDATE_THRESHOLD = 0.45   # Max min_exemplar_dist for candidates

# ========== END CONFIGURATION ==========

config = PipelineConfig(
    K=K,
    distance_threshold=DISTANCE_THRESHOLD,
    min_cluster_size=MIN_CLUSTER_SIZE,
    yaw_max=YAW_MAX,
    pitch_max=PITCH_MAX,
    roll_max=ROLL_MAX,
    blur_min=BLUR_MIN,
    d10_k=D10_K,
    exemplars_d10_threshold=EXEMPLAR_THRESHOLD,
    N_exemplars_max=10,
    exemplar_suppression_radius=0.2,
)

print(f"Configuration:")
print(f"  Embeddings: {EMBEDDINGS_FILE}")
print(f"  Output: {OUTPUT_DIR}")
print(f"  Clustering: K={K}, threshold={DISTANCE_THRESHOLD}, min_size={MIN_CLUSTER_SIZE}")
print(f"  Quality: blur_min={BLUR_MIN}, pose filtering={'enabled' if ESTIMATE_POSE else 'disabled'}")
print(f"  Candidate threshold: {CANDIDATE_THRESHOLD}")

## Load Embeddings & Metadata

In [ ]:
def get_face_crop(face_idx, crops_dir):
    """Load face crop image."""
    for pattern in [f"face_{face_idx:04d}_aligned.jpg", f"face_{face_idx:04d}.jpg"]:
        crop_path = crops_dir / pattern
        if crop_path.exists():
            return np.array(Image.open(crop_path))
    return None

# Load embeddings
print(f"Loading embeddings from: {EMBEDDINGS_FILE}")
embeddings = np.load(EMBEDDINGS_FILE)
print(f"Loaded {embeddings.shape[0]} embeddings")

# Load metadata
json_file = EMBEDDINGS_FILE.parent / EMBEDDINGS_FILE.name.replace("embeddings_", "benchmark_").replace(".npy", ".json")
if json_file.exists():
    with open(json_file) as f:
        benchmark_data = json.load(f)
    face_metadata = benchmark_data.get('face_metadata', [])
    print(f"Loaded metadata for {len(face_metadata)} faces")
else:
    print(f"Warning: metadata file not found")
    face_metadata = []

# Normalize embeddings
embeddings = embeddings / np.linalg.norm(embeddings, axis=1, keepdims=True)

# Create FaceRecord objects
faces = []
crops_dir = EMBEDDINGS_FILE.parent / "face_crops"

for i in range(len(embeddings)):
    aligned_face = get_face_crop(i, crops_dir)
    
    if i < len(face_metadata):
        meta = face_metadata[i]
        image_id = Path(meta['image_path']).name
        image_path = meta['image_path']
        face_index = meta['face_index']
        bbox = (
            meta['bbox']['x_px'],
            meta['bbox']['y_px'],
            meta['bbox']['w_px'],
            meta['bbox']['h_px']
        )
    else:
        image_id = f"face_{i:04d}"
        image_path = None
        face_index = None
        bbox = (0.0, 0.0, 112.0, 112.0)
    
    face = FaceRecord(
        face_id=i,
        image_id=image_id,
        bbox=bbox,
        aligned_face=aligned_face,
        embedding=embeddings[i],
        embedding_normalized=embeddings[i],
        pose=(0.0, 0.0, 0.0),
        blur_score=100.0,
        area=bbox[2] * bbox[3],
        is_core=False,
        image_path=image_path,
        face_index=face_index
    )
    faces.append(face)

print(f"Created {len(faces)} FaceRecord objects")
print(f"Face crops loaded: {sum(1 for f in faces if f.aligned_face is not None)} / {len(faces)}")

## Quality Gating

In [ ]:
# Initialize gater
gater = QualityGater(config, use_pose_estimation=ESTIMATE_POSE, device='cpu')

# Compute blur scores
faces = gater.compute_blur_scores(faces)

# Optionally compute pose
if ESTIMATE_POSE:
    print("Computing pose (may take a while)...")
    faces = gater.compute_pose_scores(faces)

# Filter faces
core_indices = []
holdout_indices = []

for i, face in enumerate(faces):
    passes_blur = face.blur_score >= config.blur_min
    passes_pose = True
    
    if ESTIMATE_POSE and face.pose is not None:
        yaw, pitch, roll = face.pose
        passes_pose = (
            abs(yaw) <= config.yaw_max and
            abs(pitch) <= config.pitch_max and
            abs(roll) <= config.roll_max
        )
    
    if passes_blur and passes_pose:
        core_indices.append(i)
        face.is_core = True
    else:
        holdout_indices.append(i)
        face.is_core = False

print(f"\nQuality gating results:")
print(f"  Core: {len(core_indices)} faces")
print(f"  Holdout: {len(holdout_indices)} faces")

# Show blur distribution
blur_scores = [f.blur_score for f in faces]
plt.figure(figsize=(10, 4))
plt.subplot(1, 2, 1)
plt.hist(blur_scores, bins=30, edgecolor='black')
plt.axvline(config.blur_min, color='r', linestyle='--', label=f'Threshold={config.blur_min}')
plt.xlabel('Blur Score')
plt.ylabel('Count')
plt.title('Blur Score Distribution')
plt.legend()

plt.subplot(1, 2, 2)
core_blur = [faces[i].blur_score for i in core_indices]
holdout_blur = [faces[i].blur_score for i in holdout_indices]
plt.boxplot([core_blur, holdout_blur], labels=['Core', 'Holdout'])
plt.ylabel('Blur Score')
plt.title('Blur Score by Set')
plt.tight_layout()
plt.show()

## Build kNN Graph & Cluster

In [ ]:
# Build distance matrix
print("Building distance matrix...")
graph_builder = KNNGraphBuilder(config)
distance_matrix = graph_builder.build_distance_matrix(faces, core_indices)
print(f"Distance matrix shape: {distance_matrix.shape}")

# Build mutual kNN graph
print("Building mutual kNN graph...")
graph_result = graph_builder.build_mutual_knn_graph(
    distance_matrix,
    config.K,
    config.distance_threshold
)
print(f"Graph: {graph_result.G.number_of_nodes()} nodes, {graph_result.G.number_of_edges()} edges")

# Connected components clustering
print("Clustering with connected components...")
clusterer = ConnectedComponentsClusterer(config)
cluster_result = clusterer.cluster(graph_result, core_indices)
print(f"Clustering: {cluster_result.n_clusters} clusters, {cluster_result.n_noise} noise")

# Exemplar selection
print("Selecting exemplars...")
exemplar_selector = D10ExemplarSelector(config)
cluster_result = exemplar_selector.select_exemplars(cluster_result, graph_result)
total_exemplars = sum(len(exs) for exs in cluster_result.exemplars.values())
print(f"Selected {total_exemplars} exemplars across {len(cluster_result.exemplars)} clusters")

## Compute Cluster Statistics

In [ ]:
# Compute stats for each cluster
cluster_stats = {}

for cluster_id, cluster_nodes in cluster_result.clusters.items():
    # Diameter
    if len(cluster_nodes) > 1:
        cluster_dists = distance_matrix[np.ix_(cluster_nodes, cluster_nodes)]
        diameter = float(cluster_dists.max())
    else:
        diameter = 0.0
    
    # T_A (P90 of exemplar distances)
    T_A = 0.0
    if cluster_id in cluster_result.exemplars:
        exemplar_nodes = cluster_result.exemplars[cluster_id]
        if len(exemplar_nodes) > 1:
            exemplar_dists = distance_matrix[np.ix_(exemplar_nodes, exemplar_nodes)]
            exemplar_dists_flat = exemplar_dists[np.triu_indices_from(exemplar_dists, k=1)]
            if len(exemplar_dists_flat) > 0:
                T_A = float(np.percentile(exemplar_dists_flat, 90))
    
    # Mean blur
    cluster_face_indices = [core_indices[n] for n in cluster_nodes]
    blur_scores = [faces[i].blur_score for i in cluster_face_indices]
    mean_blur = float(np.mean(blur_scores))
    
    # Face IDs
    face_ids = [faces[core_indices[n]].face_id for n in cluster_nodes]
    
    cluster_stats[cluster_id] = {
        'diameter': diameter,
        'T_A': T_A,
        'mean_blur': mean_blur,
        'face_ids': face_ids,
    }

# Compute T_global
T_A_values = [stats['T_A'] for stats in cluster_stats.values() if stats['T_A'] > 0]
T_global = float(np.median(T_A_values)) if len(T_A_values) > 0 else 0.0

print(f"Cluster statistics:")
print(f"  T_global (median T_A): {T_global:.3f}")
print(f"  Diameter range: [{min(s['diameter'] for s in cluster_stats.values()):.3f}, "
      f"{max(s['diameter'] for s in cluster_stats.values()):.3f}]")

# Visualize T_A and diameter distributions
T_A_list = [s['T_A'] for s in cluster_stats.values() if s['T_A'] > 0]
diameter_list = [s['diameter'] for s in cluster_stats.values()]

plt.figure(figsize=(12, 4))
plt.subplot(1, 2, 1)
plt.hist(T_A_list, bins=20, edgecolor='black')
plt.axvline(T_global, color='r', linestyle='--', label=f'T_global={T_global:.3f}')
plt.xlabel('T_A (P90 exemplar distance)')
plt.ylabel('Count')
plt.title('Per-Cluster Threshold Distribution')
plt.legend()

plt.subplot(1, 2, 2)
plt.hist(diameter_list, bins=20, edgecolor='black')
plt.xlabel('Cluster Diameter')
plt.ylabel('Count')
plt.title('Cluster Diameter Distribution')
plt.tight_layout()
plt.show()

## Generate Candidate Pairs

In [ ]:
# Generate candidates filtered by exemplar distance
candidates = []
cluster_ids = sorted(cluster_result.clusters.keys())

for i, cluster_id_a in enumerate(cluster_ids):
    for cluster_id_b in cluster_ids[i+1:]:
        # Get exemplars
        exemplars_a = cluster_result.exemplars.get(
            cluster_id_a,
            cluster_result.clusters[cluster_id_a]
        )
        exemplars_b = cluster_result.exemplars.get(
            cluster_id_b,
            cluster_result.clusters[cluster_id_b]
        )
        
        # Min exemplar distance
        exemplar_dists = distance_matrix[np.ix_(exemplars_a, exemplars_b)]
        min_exemplar_dist = float(exemplar_dists.min())
        
        # Filter by threshold
        if min_exemplar_dist < CANDIDATE_THRESHOLD:
            candidates.append((cluster_id_a, cluster_id_b, min_exemplar_dist))

# Sort by distance
candidates.sort(key=lambda x: x[2])

print(f"Generated {len(candidates)} candidate pairs (from {len(cluster_ids)} clusters)")
print(f"Candidate threshold: {CANDIDATE_THRESHOLD}")
print(f"Closest pair: clusters {candidates[0][0]} and {candidates[0][1]} (dist={candidates[0][2]:.3f})")

# Visualize candidate distance distribution
candidate_dists = [c[2] for c in candidates]
plt.figure(figsize=(8, 4))
plt.hist(candidate_dists, bins=20, edgecolor='black')
plt.axvline(CANDIDATE_THRESHOLD, color='r', linestyle='--', label=f'Threshold={CANDIDATE_THRESHOLD}')
plt.xlabel('Min Exemplar Distance')
plt.ylabel('Count')
plt.title('Candidate Pair Distance Distribution')
plt.legend()
plt.show()

## Compute Pair Features

Compute all 12 features for each candidate pair.

In [ ]:
def compute_pair_features(cluster_id_a, cluster_id_b):
    """Compute all 12 features for a cluster pair."""
    nodes_a = cluster_result.clusters[cluster_id_a]
    nodes_b = cluster_result.clusters[cluster_id_b]
    
    exemplars_a = cluster_result.exemplars.get(cluster_id_a, nodes_a)
    exemplars_b = cluster_result.exemplars.get(cluster_id_b, nodes_b)
    
    # 1. min_exemplar_dist
    exemplar_dists = distance_matrix[np.ix_(exemplars_a, exemplars_b)]
    min_exemplar_dist = float(exemplar_dists.min())
    
    # Cross-cluster distances
    cross_dists = distance_matrix[np.ix_(nodes_a, nodes_b)].flatten()
    
    # 2-3. p10, p50
    p10_cross_dist = float(np.percentile(cross_dists, 10))
    p50_cross_dist = float(np.percentile(cross_dists, 50))
    
    # 4. support_fraction
    support_fraction = float(np.mean(cross_dists < 0.35))
    
    # 5. diameter_ratio
    dia_a = cluster_stats[cluster_id_a]['diameter']
    dia_b = cluster_stats[cluster_id_b]['diameter']
    diameter_ratio = max(dia_a, dia_b) / min(dia_a, dia_b) if dia_a > 0 and dia_b > 0 else 1.0
    
    # 6-7. cluster sizes
    size_a = len(nodes_a)
    size_b = len(nodes_b)
    cluster_size_min = min(size_a, size_b)
    cluster_size_ratio = max(size_a, size_b) / min(size_a, size_b)
    
    # 8-11. thresholds
    T_A = cluster_stats[cluster_id_a]['T_A']
    T_B = cluster_stats[cluster_id_b]['T_A']
    T_local = max(T_A, T_B)
    
    return {
        'cluster_id_1': cluster_id_a,
        'cluster_id_2': cluster_id_b,
        'min_exemplar_dist': min_exemplar_dist,
        'p10_cross_dist': p10_cross_dist,
        'p50_cross_dist': p50_cross_dist,
        'support_fraction': support_fraction,
        'diameter_ratio': diameter_ratio,
        'cluster_size_min': cluster_size_min,
        'cluster_size_ratio': cluster_size_ratio,
        'T_A': T_A,
        'T_B': T_B,
        'T_local': T_local,
        'T_global': T_global,
    }

# Compute features for all candidates
print("Computing features for all candidate pairs...")
pair_features = []
for cluster_id_a, cluster_id_b, _ in candidates:
    features = compute_pair_features(cluster_id_a, cluster_id_b)
    pair_features.append(features)

print(f"Computed features for {len(pair_features)} pairs")

# Show feature correlations
pairs_df = pd.DataFrame(pair_features)
feature_cols = ['min_exemplar_dist', 'p10_cross_dist', 'p50_cross_dist', 'support_fraction', 
                'diameter_ratio', 'cluster_size_min', 'cluster_size_ratio', 'T_local']
corr = pairs_df[feature_cols].corr()

plt.figure(figsize=(10, 8))
plt.imshow(corr, cmap='coolwarm', vmin=-1, vmax=1)
plt.colorbar(label='Correlation')
plt.xticks(range(len(feature_cols)), feature_cols, rotation=45, ha='right')
plt.yticks(range(len(feature_cols)), feature_cols)
plt.title('Feature Correlation Matrix')
plt.tight_layout()
plt.show()

print("\nFeature summary:")
print(pairs_df[feature_cols].describe())

## Export CSVs

In [ ]:
# Create output directory
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# 1. faces.csv
face_to_cluster = {}
for cid, cluster_nodes in cluster_result.clusters.items():
    for node_idx in cluster_nodes:
        face_idx = core_indices[node_idx]
        face_to_cluster[face_idx] = cid

faces_data = []
for i, face in enumerate(faces):
    yaw, pitch, roll = face.pose if face.pose else (None, None, None)
    cluster_id = face_to_cluster.get(i, -1)
    
    faces_data.append({
        'face_id': face.face_id,
        'image_path': face.image_path or '',
        'cluster_id': cluster_id,
        'bbox_x': face.bbox[0],
        'bbox_y': face.bbox[1],
        'bbox_w': face.bbox[2],
        'bbox_h': face.bbox[3],
        'blur_score': face.blur_score,
        'pose_yaw': yaw,
        'pose_pitch': pitch,
        'pose_roll': roll,
        'is_core': face.is_core,
        'embedding_index': face.face_id,
    })

faces_df = pd.DataFrame(faces_data)
faces_path = OUTPUT_DIR / 'faces.csv'
faces_df.to_csv(faces_path, index=False)
print(f"Exported {len(faces_df)} faces to: {faces_path}")

# 2. clusters.csv
clusters_data = []
for cluster_id, cluster_nodes in cluster_result.clusters.items():
    exemplar_nodes = cluster_result.exemplars.get(cluster_id, [])
    exemplar_face_ids = [faces[core_indices[n]].face_id for n in exemplar_nodes]
    face_ids = cluster_stats[cluster_id]['face_ids']
    
    clusters_data.append({
        'cluster_id': cluster_id,
        'cluster_size': len(cluster_nodes),
        'exemplar_ids': ','.join(map(str, exemplar_face_ids)),
        'diameter': cluster_stats[cluster_id]['diameter'],
        'T_A': cluster_stats[cluster_id]['T_A'],
        'mean_blur': cluster_stats[cluster_id]['mean_blur'],
        'face_ids': ','.join(map(str, face_ids)),
    })

clusters_df = pd.DataFrame(clusters_data)
clusters_path = OUTPUT_DIR / 'clusters.csv'
clusters_df.to_csv(clusters_path, index=False)
print(f"Exported {len(clusters_df)} clusters to: {clusters_path}")

# 3. candidate_pairs.csv
pairs_df = pd.DataFrame(pair_features)
pairs_path = OUTPUT_DIR / 'candidate_pairs.csv'
pairs_df.to_csv(pairs_path, index=False)
print(f"Exported {len(pairs_df)} candidate pairs to: {pairs_path}")

# Export summary
summary = {
    'timestamp': datetime.now().isoformat(),
    'n_faces': len(faces),
    'n_core': len(core_indices),
    'n_holdout': len(faces) - len(core_indices),
    'n_clusters': cluster_result.n_clusters,
    'n_noise': cluster_result.n_noise,
    'n_candidate_pairs': len(pair_features),
    'config': {
        'K': K,
        'distance_threshold': DISTANCE_THRESHOLD,
        'min_cluster_size': MIN_CLUSTER_SIZE,
        'blur_min': BLUR_MIN,
        'candidate_threshold': CANDIDATE_THRESHOLD,
    },
    'files': {
        'faces': faces_path.name,
        'clusters': clusters_path.name,
        'pairs': pairs_path.name,
    }
}

summary_path = OUTPUT_DIR / 'export_summary.json'
with open(summary_path, 'w') as f:
    json.dump(summary, f, indent=2)

print(f"\nExport summary saved to: {summary_path}")

## Summary

Export complete! Files created:
- `faces.csv` - metadata for all faces
- `clusters.csv` - statistics for each cluster
- `candidate_pairs.csv` - features for ML training
- `export_summary.json` - export metadata

**Next steps**:
1. Use Streamlit labeling app to assign corrected_identity to each cluster
2. Run training script to generate merge_training_data.csv and train logistic regression
3. Deploy trained model as MLMerger

In [ ]:
# Display summary
print("="*60)
print("EXPORT SUMMARY")
print("="*60)
print(f"Total faces: {len(faces)}")
print(f"  Core: {len(core_indices)}")
print(f"  Holdout: {len(faces) - len(core_indices)}")
print(f"\nClusters: {cluster_result.n_clusters}")
print(f"Noise: {cluster_result.n_noise}")
print(f"\nCandidate pairs: {len(pair_features)}")
print(f"Candidate threshold: {CANDIDATE_THRESHOLD}")
print(f"\nOutput directory: {OUTPUT_DIR}")
print(f"\nFiles created:")
print(f"  - {faces_path.name} ({len(faces_df)} rows)")
print(f"  - {clusters_path.name} ({len(clusters_df)} rows)")
print(f"  - {pairs_path.name} ({len(pairs_df)} rows)")
print(f"  - {summary_path.name}")
print("\n" + "="*60)
print("Next: Use Streamlit app for manual labeling")
print("  python -m streamlit run app/face_clustering_labeling.py")
print("="*60)